# Diarios de avances de exploración
¿Cuánto aporta cada algoritmo y en qué momento conviene compartir cálculos? Cada sección ejecuta una variante en el mismo puzzle pequeño y después compara las mediciones reales con el paso anterior. Menos instrucciones es mejor.

Las comparaciones usan únicamente IDs resueltos por ambos métodos. La tabla final incluye cobertura para distinguir resultados parciales de entregas completas. Los 1.000 puntuables se miden localmente.

## Datos y funciones de lectura
Se toma la corrida completa más reciente de cada conjunto. Para regenerar los reportes desde la raíz, ejecutar `python3 -m scripts.compare_dag` (práctica) o `python3 -m scripts.scored_dag` (puntuables). Ejecutar el diario solo lee esos reportes y resuelve el ejemplo.

In [1]:
import csv
from pathlib import Path
from scripts.compare_dag import (Puzzle, validate, incremental, rings, bisection, multirings,
    bisection_reuse, multirings_reuse, postprocess, optimize_a4, nested_windows, optimize_dag)

def leer(ruta):
    with ruta.open(newline='') as archivo:
        return list(csv.DictReader(archivo))

def reporte(prefijo, marca):
    archivos = list(Path('reports').glob(f'{prefijo}_*/{marca}'))
    if not archivos:
        raise FileNotFoundError(f'Falta una corrida completa de {prefijo}. Ver comandos arriba.')
    return max(archivos, key=lambda p: p.stat().st_mtime_ns).parent

carpetas = {'Práctica': reporte('dag_practice', 'summary.csv'),
            'Puntuables': reporte('dag_scored', 'submission.csv')}
datos = {k: leer(v / 'comparison.csv') for k, v in carpetas.items()}
ejemplo = Puzzle('ejemplo', [1, 3, 10], [27, 97, 99, 107], 200)
historial = []

def medir(filas, columna, algoritmo=None, etapa=None):
    valores = {}
    for fila in filas:
        if fila['error'] or (algoritmo and fila['algorithm'] != algoritmo):
            continue
        if etapa and fila['insertion'] != etapa:
            continue
        valor = int(fila[columna])
        k = fila['id']
        valores[k] = min(valores.get(k, valor), valor)
    return valores

def comparar(nombre, columna, algoritmo=None, etapa=None):
    actual = {k: medir(filas, columna, algoritmo, etapa) for k, filas in datos.items()}
    for k, valores in actual.items():
        if historial:
            prev = historial[-1][1][k]
            comunes = prev.keys() & valores.keys()
            ahorro = sum(prev[i] - valores[i] for i in comunes)
            print(f'{k}: {len(valores)} resueltos | {len(comunes)} comunes | ahorro frente a {historial[-1][0]}: {ahorro:+,}')
        else:
            print(f'{k}: {len(valores)} resueltos | {sum(valores.values()):,} instrucciones')
    historial.append((nombre, actual))

for k, ruta in carpetas.items():
    print(k, ruta)

Práctica reports/dag_practice_gpvvc01y
Puntuables reports/dag_scored_ojhhopuu


## A1 · Incremental
Conserva un uno y suma hasta alcanzar cada objetivo en orden. Sirve como primera referencia y puede agotar el límite de instrucciones.

In [2]:
programa = incremental(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 107, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Establece la referencia inicial.

In [3]:
comparar('A1 · Incremental', 'before', 'incremental', 'constructor')

Práctica: 16 resueltos | 144 instrucciones
Puntuables: 16 resueltos | 139 instrucciones


## A2 · Anillos
Avanza con múltiplos de una base y completa el resto para alcanzar el objetivo.

In [4]:
programa = rings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [5]:
comparar('A2 · Anillos', 'before', 'rings', 'constructor')

Práctica: 159 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +8
Puntuables: 778 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +4


## A3 · Bisección
Parte del intervalo entre múltiplos y construye valores intermedios para acercarse al objetivo.

In [6]:
programa = bisection(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [7]:
comparar('A3 · Bisección', 'before', 'bisection', 'constructor')

Práctica: 142 resueltos | 133 comunes | ahorro frente a A2 · Anillos: -126,549
Puntuables: 775 resueltos | 684 comunes | ahorro frente a A2 · Anillos: -683,353


## Múltiples anillos
Explora distintas bases iniciales para acortar la construcción.

In [8]:
programa = multirings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 33, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [9]:
comparar('Múltiples anillos', 'before', 'multirings', 'constructor')

Práctica: 163 resueltos | 142 comunes | ahorro frente a A3 · Bisección: +67,763
Puntuables: 853 resueltos | 775 comunes | ahorro frente a A3 · Bisección: +369,669


## Bisección con reutilización
Aprovecha extremos y valores ya presentes en registros durante la bisección.

In [10]:
programa = bisection_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 20, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [11]:
comparar('Bisección con reutilización', 'before', 'bisection_reuse', 'constructor')

Práctica: 200 resueltos | 163 comunes | ahorro frente a Múltiples anillos: +153,313
Puntuables: 1000 resueltos | 853 comunes | ahorro frente a Múltiples anillos: +829,456


## Múltiples anillos con reutilización
Combina la elección de base con el aprovechamiento de valores residentes. Este programa alimenta las tres pasadas siguientes.

In [12]:
programa = multirings_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 14, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [13]:
comparar('Múltiples anillos con reutilización', 'before', 'multirings_reuse', 'constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Bisección con reutilización: +14,062
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Bisección con reutilización: +66,480


## Posprocesamiento
Comprime operaciones y elimina cálculos sin uso.

In [14]:
programa = postprocess(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [15]:
comparar('Posprocesamiento', 'before', 'multirings_reuse', 'postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Múltiples anillos con reutilización: +114,905
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Múltiples anillos con reutilización: +576,348


## A4
Busca atajos que permiten pasarse de un objetivo y corregir por resta o división exacta.

In [16]:
programa = optimize_a4(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [17]:
comparar('A4', 'before', 'multirings_reuse', 'a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Posprocesamiento: +550
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Posprocesamiento: +2,579


## Ventanas de ventanas
Une bloques vecinos para aprovechar valores que cruzan sus límites.

In [18]:
programa = nested_windows(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [19]:
comparar('Ventanas de ventanas', 'before', 'multirings_reuse', 'nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a A4: +437
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a A4: +2,239


## Mejor control por puzzle
Cada constructor válido recibe las tres pasadas. Se elige el programa final más corto por ID para medir el efecto de poder cambiar de constructor.

In [20]:
constructores = (incremental, rings, bisection, multirings, bisection_reuse, multirings_reuse)
pasadas = (postprocess, optimize_a4, nested_windows)

def ruta(constructor, insercion=None):
    programa = constructor(ejemplo)
    for posicion in range(4):
        if posicion == insercion:
            programa = optimize_dag(ejemplo, programa)
        if posicion < len(pasadas):
            programa = pasadas[posicion](ejemplo, programa)
    return validate(ejemplo, programa)['steps']

print(min(ruta(fn) for fn in constructores))

7


### Comparación con ventanas
Cuantifica el cambio de usar el mejor constructor por ID.

In [21]:
comparar('Mejor control sin DAG', 'control_final')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Ventanas de ventanas: +9,800
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Ventanas de ventanas: +44,318


## DAG después de constructor
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [22]:
print(min(ruta(fn, 0) for fn in constructores))

10


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [23]:
comparar('DAG después de constructor', 'dag_final', etapa='constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Mejor control sin DAG: +41,649
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Mejor control sin DAG: +194,289


## DAG después de posprocesamiento
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [24]:
print(min(ruta(fn, 1) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [25]:
comparar('DAG después de posprocesamiento', 'dag_final', etapa='postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de constructor: -26,811
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de constructor: -122,126


## DAG después de A4
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [26]:
print(min(ruta(fn, 2) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [27]:
comparar('DAG después de A4', 'dag_final', etapa='a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de posprocesamiento: -77
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de posprocesamiento: -499


## DAG después de ventanas
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [28]:
print(min(ruta(fn, 3) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [29]:
comparar('DAG después de ventanas', 'dag_final', etapa='nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de A4: +0
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de A4: +0


## Selección final por puzzle
Se conserva el mejor de los controles y las cuatro posiciones del DAG. Para el entregable también compite el programa del CSV previo en cada ID.

In [30]:
seleccion = {}
for k, filas in datos.items():
    control = medir(filas, 'control_final')
    dag = medir(filas, 'dag_final')
    seleccion[k] = {i: min(valor, dag[i]) for i, valor in control.items()}
ganadores = leer(carpetas['Puntuables'] / 'winners.csv')
entrega = {r['id']: int(r['steps']) for r in ganadores}
anterior = {r['id']: int(r['previous']) for r in ganadores}
assert len(entrega) == 1000 and len(seleccion['Práctica']) == 200
assert entrega == {i: min(anterior[i], seleccion['Puntuables'][i]) for i in entrega}
print('Entregable validado:', carpetas['Puntuables'] / 'submission.csv')

Entregable validado: reports/dag_scored_ojhhopuu/submission.csv


### Comparación con la entrega anterior
Cuenta cuántos puzzles mejoraron y cuánto bajó la suma de instrucciones.

In [31]:
print('Puzzles mejorados:', sum(entrega[i] < anterior[i] for i in entrega))
print('Instrucciones ahorradas:', sum(anterior.values()) - sum(entrega.values()))

Puzzles mejorados: 978
Instrucciones ahorradas: 250357


## Tabla de avances
Totales de instrucciones calculados desde los reportes seleccionados. Cada fila indica cuántos puzzles se resolvieron sobre 200 o 1.000; una suma parcial solo describe esos IDs. El signo — indica que esa entrega solo existe para los puntuables.

In [32]:
def celda(valores, cantidad):
    if not valores:
        return '—'
    total = f'{sum(valores.values()):,}'.replace(',', '.')
    return f'{total} ({len(valores)}/{cantidad})'

filas = historial + [
    ('Mejor control o DAG por ID', seleccion),
    ('Entregable anterior', {'Puntuables': anterior}),
    ('Entregable validado', {'Puntuables': entrega})]
from html import escape
from IPython.display import HTML, display

encabezados = ('Algoritmo o entrega', 'Práctica · pasos (cobertura)', 'Puntuables · pasos (cobertura)')
tabla = '<table><thead><tr>' + ''.join(f'<th>{escape(t)}</th>' for t in encabezados) + '</tr></thead><tbody>'
for nombre, valores in filas:
    columnas = (nombre, celda(valores.get('Práctica'), 200), celda(valores.get('Puntuables'), 1000))
    tabla += '<tr>' + ''.join(f'<td>{escape(v)}</td>' for v in columnas) + '</tr>'
display(HTML(tabla + '</tbody></table>'))

Algoritmo o entrega,Práctica · pasos (cobertura),Puntuables · pasos (cobertura)
A1 · Incremental,144 (16/200),139 (16/1000)
A2 · Anillos,234.714 (159/200),1.166.626 (778/1000)
A3 · Bisección,332.967 (142/200),1.929.943 (775/1000)
Múltiples anillos,340.301 (163/200),1.834.813 (853/1000)
Bisección con reutilización,281.552 (200/200),1.372.573 (1000/1000)
Múltiples anillos con reutilización,267.490 (200/200),1.306.093 (1000/1000)
Posprocesamiento,152.585 (200/200),729.745 (1000/1000)
A4,152.035 (200/200),727.166 (1000/1000)
Ventanas de ventanas,151.598 (200/200),724.927 (1000/1000)
Mejor control sin DAG,141.798 (200/200),680.609 (1000/1000)
